# Robinhood User Churn — Data Audit

**Purpose:** summarize the supplied datasets and validate their structure before labeling or modeling.

Business questions:
1. What percentage of users churned?
2. Can a classifier predict churn probabilities?
3. Which features relate to churn?

This notebook prepares the evidence for those questions. Churn rates and predictive performance require later labeling and modeling steps. Raw data and user-level outputs are not committed.

**Requirements:** Python, pandas, and a Jupyter kernel (ipykernel). Place both supplied CSV files in the repository's `data/` directory. Run cells in order.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the project whether the kernel starts in the root or notebooks/.
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data/features_data.csv').is_file()
     and (p / 'data/equity_value_data.csv').is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError('Place both CSV files in the project data/ folder and open this notebook from the project.')

features = pd.read_csv(ROOT / 'data/features_data.csv', dtype={'user_id': 'string'})
equity = pd.read_csv(ROOT / 'data/equity_value_data.csv', dtype={'user_id': 'string'})
required = {'user_id', 'timestamp', 'close_equity'}
if not required.issubset(equity.columns) or 'user_id' not in features.columns:
    raise ValueError('Unexpected input schema; check the supplied CSV files.')
equity['timestamp'] = pd.to_datetime(equity['timestamp'], utc=True, errors='raise')
equity['close_equity'] = pd.to_numeric(equity['close_equity'], errors='raise')


## 1. Dataset size and quality

Summarize existing records. Zero null values do not imply complete calendar coverage.


In [ ]:
overview = pd.DataFrame([
    {'dataset': name, 'rows': len(df), 'columns': df.shape[1],
     'unique_users': df['user_id'].nunique(),
     'missing_values': int(df.isna().sum().sum()),
     'duplicate_rows': int(df.duplicated().sum())}
    for name, df in {'Features': features, 'Equity': equity}.items()
])
display(overview)
for name, df in {'Features': features, 'Equity': equity}.items():
    print(name)
    display(pd.DataFrame({'data_type': df.dtypes.astype(str),
                          'missing_count': df.isna().sum(),
                          'unique_values': df.nunique()}))


## 2. User alignment and record grain

Use `user_id` as a join key, not as a model predictor. Do not join the daily panel directly to features without defining the intended output grain.


In [ ]:
feature_users = set(features['user_id'].dropna())
equity_users = set(equity['user_id'].dropna())
alignment = pd.Series({
    'users_in_both': len(feature_users & equity_users),
    'features_only': len(feature_users - equity_users),
    'equity_only': len(equity_users - feature_users),
    'duplicate_feature_user_ids': int(features['user_id'].duplicated().sum()),
    'duplicate_equity_user_dates': int(equity.duplicated(['user_id', 'timestamp']).sum())
}, name='count')
display(alignment.to_frame())
if features['user_id'].isna().any() or equity['user_id'].isna().any():
    raise ValueError('Missing user IDs must be resolved before labeling.')
if equity.duplicated(['user_id', 'timestamp']).any():
    raise ValueError('Resolve duplicate user-date records before analyzing coverage.')


## 3. Numeric distributions

Review medians, spread, extreme values, and negative values before choosing transformations. The units and measurement window of `time_spent` still require documentation.


In [ ]:
display(features.select_dtypes(include='number').describe().T.round(2))
display(equity[['close_equity']].describe().T.round(2))


## 4. Categorical distributions

Counts and percentages describe the feature population; they do not yet establish associations with churn.


In [ ]:
categorical_columns = features.select_dtypes(include=['object', 'string', 'category']).columns.drop('user_id', errors='ignore')
for column in categorical_columns:
    print(column)
    summary = features[column].value_counts(dropna=False).to_frame('users')
    summary['percentage'] = (100 * summary['users'] / len(features)).round(2)
    display(summary)


## 5. Date coverage and the reporting threshold

The supplied lesson states that records appear only when equity is at least $10. Below-threshold states therefore appear as absent records, rather than numeric values below $10. The absence of all weekend records introduces a reporting-calendar assumption that must be made explicit before labeling.


In [ ]:
display(pd.Series({
    'observation_start': equity['timestamp'].min(),
    'observation_end': equity['timestamp'].max(),
    'distinct_observed_dates': equity['timestamp'].nunique(),
    'minimum_recorded_equity': equity['close_equity'].min(),
    'rows_below_10': int(equity['close_equity'].lt(10).sum())
}, name='value').to_frame())
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
display(equity['timestamp'].dt.day_name().value_counts()
        .reindex(weekday_order, fill_value=0).to_frame('records'))


## 6. Coverage and gaps per user

Internal absent days exclude the first and last observed dates. They include weekends and other missing dates. They are a coverage measure, not yet a churn label. A trailing gap is measured only up to the observed dataset end; no future days are invented.


In [ ]:
ordered = equity.sort_values(['user_id', 'timestamp']).copy()
coverage = ordered.groupby('user_id').agg(
    first_observed=('timestamp', 'min'),
    last_observed=('timestamp', 'max'),
    observed_days=('timestamp', 'nunique')
)
coverage['calendar_span_days'] = (coverage['last_observed'] - coverage['first_observed']).dt.days + 1
coverage['missing_days_within_span'] = coverage['calendar_span_days'] - coverage['observed_days']
coverage['trailing_absent_days'] = (equity['timestamp'].max() - coverage['last_observed']).dt.days
ordered['internal_absent_days'] = ordered.groupby('user_id')['timestamp'].diff().dt.days - 1
coverage['longest_internal_gap_days'] = ordered.groupby('user_id')['internal_absent_days'].max().reindex(coverage.index).fillna(0).astype(int)
summary_columns = ['observed_days', 'calendar_span_days', 'missing_days_within_span',
                   'longest_internal_gap_days', 'trailing_absent_days']
display(coverage[summary_columns].describe().T.round(2))
print('Users with at least one internal absent day:', int(coverage['missing_days_within_span'].gt(0).sum()))


## 7. Findings and next decisions

Record facts separately from assumptions. No user-level records are displayed or saved by this notebook.


In [ ]:
print(f'Both datasets share {len(feature_users & equity_users):,} users.')
print(f'Equity records: {len(equity):,}; minimum recorded equity: ${equity.close_equity.min():.2f}.')
print(f'Recorded rows below $10: {equity.close_equity.lt(10).sum():,}.')
print(f'Median internal absent days per user: {coverage.missing_days_within_span.median():.0f}.')
print('Next: document calendar handling, label 28-day spells, then compare features by churn outcome.')


### How this supports the business questions

| Question | Next step |
| --- | --- |
| Percentage churned | Define calendar-gap assumptions, count qualifying internal and trailing spells after prior equity >= $10, and state the denominator. |
| Predict churn | Establish feature availability and outcome windows; fit and evaluate a classifier on unseen users/data. |
| Features related to churn | Compare groups after labeling and assess feature rankings; associations do not prove causation. |

**Label boundaries:** between two recorded dates, absent calendar days equal the date difference minus one. After the final record, absent days equal dataset end minus final record. Before first recorded equity >= $10, missing days do not satisfy the prior-equity condition. Treat these as candidate spells until reporting-calendar assumptions are fixed.

**Prediction limitation:** profile fields have no measurement timestamps in the supplied feature file. Confirm their availability at prediction time, especially `time_spent`, before claiming prospective prediction.

Keep saved notebook outputs clear before committing if you add cells that display identifiers or user-level data.
